# 01 — Il dataset del tentativo 13

Il dataset **solo politici** del tentativo 12 (testate con il solo argomento "Political news or commentary" per
NewsGuard 2026, etichetta = livello NewsGuard 2026, due giri di ottobre + terzo giro), **senza le testate con meno di
10 articoli** (quasi tutte di una rete di siti l1 con circa 7 articoli ciascuno, che nel test non entravano). Le testate
valutate (almeno 20 articoli) e i fold restano quelli del tentativo 12; il peso di solo training si ricalcola,
min(1, 5/n) con n le testate del gruppo che restano. Per l'analisi per genere si aggiungono orientamento e paese
NewsGuard 2026 (solo per descrivere, mai come ingresso dei modelli).

In [ ]:
import numpy as np
import pandas as pd

DATASET, RESULTS = "../../../../../Dataset/tentativi/13_XAI", "../Results/1-dataset"
import os; os.makedirs(RESULTS, exist_ok=True); os.makedirs(f"{DATASET}/versioni", exist_ok=True)
MIN_ARTICOLI_TRAINING = 10
MAX_TESTATE_PER_GRUPPO = 5
LIVELLI = ["l1", "l2", "l3", "l4", "l5"]
art = pd.read_csv(f"{DATASET}/versioni_t12/dataset.csv")
t = pd.read_csv(f"{DATASET}/versioni_t12/fold.csv").set_index("domain")
print(f"tentativo 12: {len(t)} testate ({t.valutazione.sum()} valutate), {len(art)} articoli")
via = t.index[t.articoli < MIN_ARTICOLI_TRAINING]
assert not t.valutazione[via].any()
print(f"via {len(via)} testate con meno di {MIN_ARTICOLI_TRAINING} articoli: livelli {t.label[via].map(dict(enumerate(LIVELLI))).value_counts().to_dict()}")
t = t.drop(via); art = art[art.domain.isin(t.index)].copy()
t["testate_nel_gruppo"] = t.groupby("gruppo").label.transform("size")
t["peso"] = np.minimum(1.0, MAX_TESTATE_PER_GRUPPO / t.testate_nel_gruppo)
art["peso"] = art.domain.map(t.peso)
ng = pd.read_csv("../../../../../Dataset/etichette/newsguard_2026.csv", low_memory=False)
ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
t["orientamento"] = t.index.map(ng.Orientation).fillna("nessuno")
paese = pd.Series(t.index.map(ng.Country), index=t.index)
t["paese"] = paese.where(paese.isin(["US", "GB", "CA", "AU"]), "altro")
art.to_csv(f"{DATASET}/versioni/dataset.csv", index=False)
t.reset_index().to_csv(f"{DATASET}/versioni/fold.csv", index=False)
comp = pd.crosstab([t.valutazione.map({True: "valutate", False: "solo training"}), t.orientamento], t.label.map(dict(enumerate(LIVELLI))), margins=True)
comp.to_csv(f"{RESULTS}/01_composizione.csv")
print(f"tentativo 13: {len(t)} testate ({t.valutazione.sum()} valutate), {len(art)} articoli; peso per livello "
      f"{t.groupby('label').peso.sum().round(1).to_dict()}")
comp